In [61]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
import resend
import os
from openai.types.responses import ResponseTextDeltaEvent
import asyncio


load_dotenv(override=True)


True

In [63]:
# inspect Agent signature only
import inspect
print("Agent init signature:", inspect.signature(Agent))
print("Attributes:", [attr for attr in dir(Agent) if not attr.startswith('_')])


Agent init signature: (name: 'str', handoff_description: 'str | None' = None, tools: 'list[Tool]' = <factory>, mcp_servers: 'list[MCPServer]' = <factory>, mcp_config: 'MCPConfig' = <factory>, instructions: 'str | Callable[[RunContextWrapper[TContext], Agent[TContext]], MaybeAwaitable[str]] | None' = None, prompt: 'Prompt | DynamicPromptFunction | None' = None, handoffs: 'list[Agent[Any] | Handoff[TContext, Any]]' = <factory>, model: 'str | Model | None' = None, model_settings: 'ModelSettings' = <factory>, input_guardrails: 'list[InputGuardrail[TContext]]' = <factory>, output_guardrails: 'list[OutputGuardrail[TContext]]' = <factory>, output_type: 'type[Any] | AgentOutputSchemaBase | None' = None, hooks: 'AgentHooks[TContext] | None' = None, tool_use_behavior: "Literal['run_llm_again', 'stop_on_first_tool'] | StopAtTools | ToolsToFinalOutputFunction" = 'run_llm_again', reset_tool_choice: 'bool' = True) -> None
Attributes: ['as_tool', 'clone', 'get_all_tools', 'get_mcp_tools', 'get_prompt

In [16]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."


sales_agent1= Agent(
    name="profesional slaes agent",
    instructions=instructions1,
    model="gpt-4o-mini"
)

sales_agent2 = Agent(
    name="Engaging sales agent",
    instructions=instructions2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="busy sales agent",
    instructions=instructions3,
    model="gpt-4o-mini"
)

In [17]:
descriptions = "write a cold sales email"

tool1= sales_agent1.as_tool(tool_name="sales_agent1", tool_description=descriptions)
tool2= sales_agent2.as_tool(tool_name="sales_agent2", tool_description= descriptions)
tool3= sales_agent3.as_tool(tool_name="sales_agent3", tool_description=descriptions)


tools= [tool1,tool2,tool3]


### Handoffs represent a way an agent can delegate to an agent, passing control to it

Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

With handoffs, control passes across



In [18]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."


subject_writer= Agent(name="Subject witer", instructions=subject_instructions, model="gpt-4o-mini")
subject_writer_tool= subject_writer.as_tool(tool_name="sunejct_writer", tool_description="Write a subject for a cold sales email")

html_converter= Agent(name="html_converter", instructions=html_instructions, model="gpt-4o-mini")
html_converter_tool= html_converter.as_tool(tool_name="html_converter", tool_description="Convert a text email body to an HTML email body")

In [50]:
@function_tool
def send_email(subject: str, html_body: str) -> str:
    """Send out an email with the given subject and HTML body to all sales prospects using Resend"""
    
    from_email = "onboarding@resend.dev"
    to_email = "shahid739815@gmail.com"
    
    resend.api_key = os.environ.get("RESEND_API_KEY")
    
    print(f"Sending email...")
    print(f"From: {from_email}")
    print(f"To: {to_email}")
    print(f"Subject: {subject}")
    print(f"API Key present: {bool(resend.api_key)}")
    
    try:
        params = {
            "from": from_email,
            "to": [to_email],
            "subject": subject,
            "html": html_body
        }
        
        email = resend.Emails.send(params)
        print(f"Response: {email}")
        
        if email.get("id"):
            return f"Email sent successfully with ID: {email['id']}"
        else:
            return f"Email failed to send: {email}"
    except Exception as e:
        print(f"Error: {e}")
        return f"Email failed to send due to error: {str(e)}"

In [52]:
# quick manual test of send_email tool object
print(send_email)
print(dir(send_email))

# the underlying function is likely available as send_email.func or .fn
try:
    print(send_email.func)
    print(send_email.func("Test subject","<p>Hello</p>"))
except Exception as e:
    print("error calling underlying function", e)

FunctionTool(name='send_email', description='Send out an email with the given subject and HTML body to all sales prospects using Resend', params_json_schema={'properties': {'subject': {'title': 'Subject', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'html_body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000002B7EF58C680>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)
['__annotations__', '__class__', '__dataclass_fields__', '__dataclass_params__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__match_args__', '__module__', '__ne__', '__new__', '__post_init__', '__reduce__', '__reduce_ex__', '__repr__'

In [28]:
handoff_tools= [subject_writer_tool,html_converter_tool,send_email]
tools


[FunctionTool(name='sales_agent1', description='write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000002B7ED4FD300>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent2', description='write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000002B7ED467920>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent3', descr

In [29]:
handoff_tools

[FunctionTool(name='sunejct_writer', description='Write a subject for a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sunejct_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000002B7EF4E3CE0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000002B7EF58C040>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=No

In [ ]:
instructions = """You are an email formatter and sender. You receive a plain text email body.

1. Call the `subject_writer` tool with the body text. Treat its output verbatim as the email subject.
2. Call the `html_converter` tool with the body text. Treat its output verbatim as the `html_body`.
3. Immediately call the `send_email` tool **with only two arguments**: the subject and the html_body. **Do not ask for or add any additional information** (to-address, from-address, confirmations, etc. are handled by the tool).
4. After invoking `send_email`, stop and return only the tool's result string. Do not include any surrounding commentary or questions.

Failure to follow these steps exactly will result in the request being handled incorrectly.
"""

email_agent = Agent(
    name="email_manager",
    instructions=instructions,
    model="gpt-4o-mini",
    tools=handoff_tools,             # use tools instead of handoffs
    handoff_description="Convert an email to HTML and send it",
    tool_use_behavior="stop_on_first_tool",
)


In [42]:
handoff=[email_agent]
handoff

[Agent(name='email_manager', handoff_description='Convert an email to HTML and send it', tools=[], mcp_servers=[], mcp_config={}, instructions='You are an email formatter and sender. You receive the body of an email to be sent.\n\nYou first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML.\n\nFinally, you use the send_email tool to send the email with the subject and HTML body.\n', prompt=None, handoffs=[FunctionTool(name='sunejct_writer', description='Write a subject for a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sunejct_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000002B7EF4E3CE0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='ht

In [ ]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""

sales_manager_agent= Agent(
    name="sales_manger",
    instructions=sales_manager_instructions,
    handoffs=handoff,
    model="gpt-4o-mini"
)

# provide a complete email body so the manager can generate drafts
message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("automate SDR"):
    result= await Runner.run(sales_manager_agent, message)